# SymPyによるベクトル解析：勾配・発散・回転の記号的計算

## 概要
電磁気学や流体力学において、ベクトル解析は言語そのものである。勾配（Gradient）、発散（Divergence）、回転（Curl）といった演算子は、場の性質を記述するために不可欠である。

通常、これらの演算は偏微分を用いて定義され、手計算で行うと計算量が多くなりがちである。Pythonの数式処理ライブラリ **SymPy** には、ベクトル解析専用のモジュール `sympy.vector` が用意されており、座標系に依存しない形式でベクトル場を定義し、各種演算を記号的に実行することができる。

本記事では、SymPyを用いて以下のトピックを解説する。

1. **ベクトル場の定義**：座標系の設定とスカラー場・ベクトル場
2. **微分演算子**：勾配 $\nabla f$、発散 $\nabla \cdot \mathbf{F}$、回転 $\nabla \times \mathbf{F}$
3. **ポテンシャル論**：保存場からのスカラーポテンシャルの導出

### 筆者の環境
筆者の実行環境は以下の通りである。

In [ ]:
!sw_vers

In [ ]:
!python -V

必要なライブラリをインポートする。`sympy.vector` からは `CoordSys3D`（3次元座標系）や各種演算子を読み込む。

In [ ]:
import sympy
from sympy import symbols, sin, cos, exp, simplify, integrate, diff
from sympy.vector import CoordSys3D, gradient, divergence, curl, scalar_potential
from pprint import pprint as py_pprint

# 数式をLaTeX形式で綺麗に表示するための設定
sympy.init_printing()

print("sympy version :", sympy.__version__)

## 1. ベクトル場の定義

SymPyでベクトル解析を行うには、まず座標系を定義する必要がある。ここでは標準的なデカルト座標系 $R$ を定義する。

In [ ]:
R = CoordSys3D('R')

# 座標変数 (x, y, z) を取得
x, y, z = R.x, R.y, R.z

# 基底ベクトル (i, j, k) を取得
i, j, k = R.i, R.j, R.k

### 1.1 スカラー場とベクトル場

スカラー場 $f(x, y, z)$ とベクトル場 $\mathbf{F}(x, y, z)$ を以下のように定義する。

$$ f = x^2 y + z $$
$$ \mathbf{F} = P \mathbf{i} + Q \mathbf{j} + R \mathbf{k} = (y z) \mathbf{i} + (x z) \mathbf{j} + (x y) \mathbf{k} $$

In [ ]:
f = x**2 * y + z
F = y*z*i + x*z*j + x*y*k

print("スカラー場 f:")
sympy.pprint(f)

print("\nベクトル場 F:")
sympy.pprint(F)

## 2. 微分演算子

ベクトル解析の基本演算である勾配、発散、回転を計算する。

### 2.1 勾配 (Gradient)

スカラー場 $f$ の勾配 $\nabla f$ はベクトル場となる。

$$ \nabla f = \frac{\partial f}{\partial x}\mathbf{i} + \frac{\partial f}{\partial y}\mathbf{j} + \frac{\partial f}{\partial z}\mathbf{k} $$

In [ ]:
grad_f = gradient(f)

print("勾配 grad(f):")
sympy.pprint(grad_f)

結果は $2xy \mathbf{i} + x^2 \mathbf{j} + \mathbf{k}$ となる。手計算での偏微分と一致することを確認できる。

### 2.2 発散 (Divergence)

ベクトル場 $\mathbf{F}$ の発散 $\nabla \cdot \mathbf{F}$ はスカラー場となる。

$$ \nabla \cdot \mathbf{F} = \frac{\partial P}{\partial x} + \frac{\partial Q}{\partial y} + \frac{\partial R}{\partial z} $$

In [ ]:
div_F = divergence(F)

print("発散 div(F):")
sympy.pprint(div_F)

今回定義した $\mathbf{F}$ の各成分は $x, y, z$ をそれぞれ含んでいないため、発散は 0 となる。これは $\mathbf{F}$ が「ソレノイダル場（管状ベクトル場）」であることを意味する。

### 2.3 回転 (Curl)

ベクトル場 $\mathbf{F}$ の回転 $\nabla \times \mathbf{F}$ はベクトル場となる。

$$ \nabla \times \mathbf{F} = \left(\frac{\partial R}{\partial y} - \frac{\partial Q}{\partial z}\right)\mathbf{i} + \left(\frac{\partial P}{\partial z} - \frac{\partial R}{\partial x}\right)\mathbf{j} + \left(\frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}\right)\mathbf{k} $$

In [ ]:
curl_F = curl(F)

print("回転 curl(F):")
sympy.pprint(curl_F)

回転も 0 となった。これは $\mathbf{F}$ が「保存場（渦なし場）」であることを意味する。
回転が 0 であるベクトル場は、あるスカラーポテンシャル $\phi$ の勾配として表せるはずである（$\mathbf{F} = \nabla \phi$）。次節でこれを検証する。

### 2.4 ベクトル解析の恒等式

ベクトル解析には重要な恒等式が存在する。例えば、「勾配の回転は常に0である」というものである。

$$ \nabla \times (\nabla f) = \mathbf{0} $$

これをSymPyで確認する。

In [ ]:
curl_grad_f = curl(gradient(f))

print("curl(grad(f)):")
sympy.pprint(curl_grad_f)

確かにゼロベクトルとなった。

## 3. ポテンシャル論

保存場 $\mathbf{F}$ に対して、$\mathbf{F} = \nabla \phi$ となるスカラーポテンシャル $\phi$ を求める問題を考える。
先ほどの $\mathbf{F} = yz \mathbf{i} + xz \mathbf{j} + xy \mathbf{k}$ は回転が0であったため、ポテンシャルが存在するはずである。

SymPyの `scalar_potential` 関数を用いれば、これを直接求めることができる。

In [ ]:
# R は定義領域（ここでは全空間）
phi = scalar_potential(F, R)

print("スカラーポテンシャル phi:")
sympy.pprint(phi)

結果は $xyz$ となった。実際に勾配をとって確認してみる。

In [ ]:
grad_phi = gradient(phi)

print("grad(phi):")
sympy.pprint(grad_phi)

print("\n元のベクトル場 F と一致するか:")
print(grad_phi == F)

一致することが確認できた。

### 3.1 手計算による導出の再現

`scalar_potential` は便利だが、内部で何が行われているかを知るために、積分による導出過程を追ってみる。

$$ \frac{\partial \phi}{\partial x} = P = yz $$
$$ \frac{\partial \phi}{\partial y} = Q = xz $$
$$ \frac{\partial \phi}{\partial z} = R = xy $$

まず第1式を $x$ で積分する。

In [ ]:
P = F.dot(i)
Q = F.dot(j)
R_comp = F.dot(k)

# xで積分
phi_step1 = integrate(P, x)

print("ステップ1 (x積分):")
sympy.pprint(phi_step1)

ここで積分定数は $y, z$ の関数 $C_1(y, z)$ である。次にこれを $y$ で偏微分し、$Q$ と比較する。

$$ \frac{\partial}{\partial y} (xyz + C_1(y, z)) = xz + \frac{\partial C_1}{\partial y} = Q = xz $$

よって $\frac{\partial C_1}{\partial y} = 0$、つまり $C_1$ は $y$ に依存しない関数 $C_2(z)$ である。
最後に $z$ で偏微分し、$R$ と比較する。

$$ \frac{\partial}{\partial z} (xyz + C_2(z)) = xy + \frac{d C_2}{d z} = R = xy $$

よって $\frac{d C_2}{d z} = 0$、$C_2$ は純粋な定数である。したがって $\phi = xyz$ が導かれる。

SymPyを用いれば、このような偏微分方程式の積分過程も、`integrate` と `diff` を組み合わせて検証することができる。

## 結論

この記事では、SymPyの `sympy.vector` モジュールを用いて、ベクトル解析の基本的な演算とポテンシャルの導出を行った。

勾配、発散、回転といった演算子は、物理学の至る所に現れる。これらを記号的に扱えることは、マクスウェル方程式の変形や流体のナビエ・ストークス方程式の解析など、より高度な物理数学の問題に取り組む際の強力な武器となる。

### 参考文献
- [SymPy Documentation: Vector Module](https://docs.sympy.org/latest/modules/vector/index.html)